# Lab - Classification

In this lab, we are going to build a classification module. When given an image of a handwritten digit like the one below, the model will be able to tell which digit is in the image.

<img src='test2.jpg'>

In [1]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier  # MLP is an NN
from sklearn import svm
import numpy as np
import argparse
import imutils  # If you are unable to install this library, ask the TA; we only need this in extract_hsv_histogram.
import cv2
import os
import random


# Depending on library versions on your system, one of the following imports 
from sklearn.model_selection import train_test_split
#from sklearn.cross_validation import train_test_split

In [2]:
path_to_dataset = r'digits_dataset'
target_img_size = (32, 32) # fix image size because classification algorithms THAT WE WILL USE HERE expect that

# We are going to fix the random seed to make our experiments reproducible 
# since some algorithms use pseudorandom generators
random_seed = 42  
random.seed(random_seed)
np.random.seed(random_seed)

## Part I - Feature Extraction

In this part, we are going to implement three functions. Each one will extract a different set of features from the image. The three sets are:

1. Histogram of the pixel values features (this is the histogram you know, but on the HSV channels)
2. Histogram of Gradients (HoG) features
3. Raw pixels (basically, not doing any feature extraction and just supplying the input image to the classifier)

In [3]:
def extract_hsv_histogram(img):
    """
    TODO
    1. Resize the image to target_img_size using cv2.resize
    2. Convert the image from BGR representation (cv2 is BGR not RGB) to HSV using cv2.cvtColor
    3. Acquire the histogram using the cv2.calcHist. Apply the functions on the 3 channels. For the bins 
        parameter pass (8, 8, 8). For the ranges parameter pass ([0, 180, 0, 256, 0, 256]). Name the histogram
        <hist>.
    """
    
    target_img_size = (128, 128)
    img = cv2.resize(img, target_img_size)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([img], [0, 1, 2], None, (8, 8, 8), [0, 180, 0, 256, 0, 256])
        
    if imutils.is_cv2():
        hist = cv2.normalize(hist)
    else:
        cv2.normalize(hist, hist)
    return hist.flatten()     

In [4]:
def extract_hog_features(img):
    """
    TODO
    You won't implement anything in this function. You just need to understand it 
    and understand its parameters (i.e win_size, cell_size, ... etc)
    """
    img = cv2.resize(img, target_img_size)
    win_size = (32, 32)
    cell_size = (4, 4)
    block_size_in_cells = (2, 2)
    
    block_size = (block_size_in_cells[1] * cell_size[1], block_size_in_cells[0] * cell_size[0])
    block_stride = (cell_size[1], cell_size[0])
    nbins = 9  # Number of orientation bins
    hog = cv2.HOGDescriptor(win_size, block_size, block_stride, cell_size, nbins)
    h = hog.compute(img)
    h = h.flatten()
    return h.flatten()

In [5]:
def extract_raw_pixels(img):
    """
    TODO
    The classification algorithms we are going to use expect the input to be a vector not a matrix. 
    This is because they are general purpose and don't work only on images.
    CNNs, on the other hand, expect matrices since they operate on images and exploit the 
    arrangement of pixels in the 2-D space.
    
    So, what we only need to do in this function is to resize and flatten the image.
    """
    img = cv2.resize(img, target_img_size)
    return img.flatten()

In [6]:
def extract_features(img, feature_set='hog'):
    """
    TODO
    Given either 'hsv_hist', 'hog', 'raw', call the respective function and return its output
    """
    if feature_set == 'hog':
        return extract_hog_features(img)
    elif feature_set == 'hsv_hist':
        return extract_hsv_histogram(img)
    elif feature_set == 'raw':
        return extract_raw_pixels(img)
    else:
        raise ValueError("Unknown feature set: {}".format(feature_set))

The following function will extract the features and the label of each image in our dataset and save it in RAM. We normally don't save datasets in RAM, but this dataset is small.

In [7]:
def load_dataset(feature_set='hog'):
    features = []
    labels = []
    img_filenames = os.listdir(path_to_dataset)

    for i, fn in enumerate(img_filenames):
        if fn.split('.')[-1] != 'jpg':
            continue

        label = fn.split('.')[0]
        labels.append(label)

        path = os.path.join(path_to_dataset, fn)
        img = cv2.imread(path)
        features.append(extract_features(img, feature_set))
        
        # show an update every 1,000 images
        if i > 0 and i % 1000 == 0:
            print("[INFO] processed {}/{}".format(i, len(img_filenames)))
        
    return features, labels        

## Part II - Classification

In this part, we will test the classification performance of SVM, KNN, & NNs given our features.

In [8]:
# TODO understand the hyperparameters of each classifier
classifiers = {
    'SVM': svm.LinearSVC(random_state=random_seed),
    'KNN': KNeighborsClassifier(n_neighbors=7),
    'NN': MLPClassifier(solver='sgd', random_state=random_seed, hidden_layer_sizes=(500,), max_iter=20, verbose=1)
}

# understanding the hyperparameters of each classifier
# SVM 
# - random_state: controls the shuffling applied to the data before applying the split
# KNN
# - n_neighbors: number of neighbors to use by default for kneighbors queries
# NN
# - solver: the algorithm for weight optimization
# - random_state: controls the randomness of the weight initialization
# - hidden_layer_sizes: the number of neurons in the hidden layers
# - max_iter: maximum number of iterations


In [9]:
# This function will test all our classifiers on a specific feature set
def run_experiment(feature_set):
    
    # Load dataset with extracted features
    print('Loading dataset. This will take time ...')
    features, labels = load_dataset(feature_set)
    print('Finished loading dataset.')
    
    # Since we don't want to know the performance of our classifier on images it has seen before
    # we are going to withhold some images that we will test the classifier on after training 
    train_features, test_features, train_labels, test_labels = train_test_split(
        features, labels, test_size=0.2, random_state=random_seed)
    
    for model_name, model in classifiers.items():
        print('############## Training', model_name, "##############")
        # Train the model only on the training features
        model.fit(train_features, train_labels)
        
        # Test the model on images it hasn't seen before
        accuracy = model.score(test_features, test_labels)
        
        print(model_name, 'accuracy:', accuracy*100, '%')

Now, we see how each classifier and each feature set performs

In [11]:
run_experiment('hog')
"""
You should get the following test accuracies the first time 

SVM accuracy ~ 97.70833333333333
KNN accuracy ~ 96.52777777777779
NN accuracy ~ 93.95833333333333
"""

Loading dataset. This will take time ...
[INFO] processed 1000/7200
[INFO] processed 2000/7200
[INFO] processed 3000/7200
[INFO] processed 4000/7200
[INFO] processed 5000/7200
[INFO] processed 6000/7200
[INFO] processed 7000/7200
Finished loading dataset.
############## Training SVM ##############
SVM accuracy: 97.63888888888889 %
############## Training KNN ##############
KNN accuracy: 96.38888888888889 %
############## Training NN ##############
Iteration 1, loss = 2.15737254
Iteration 2, loss = 1.99592821
Iteration 3, loss = 1.83601414
Iteration 4, loss = 1.68281986
Iteration 5, loss = 1.53496454
Iteration 6, loss = 1.39564823
Iteration 7, loss = 1.26638803
Iteration 8, loss = 1.14923912
Iteration 9, loss = 1.04489513
Iteration 10, loss = 0.95292445
Iteration 11, loss = 0.87254067
Iteration 12, loss = 0.80252523
Iteration 13, loss = 0.74133567
Iteration 14, loss = 0.68789341
Iteration 15, loss = 0.64135879
Iteration 16, loss = 0.60053676
Iteration 17, loss = 0.56462017
Iteration 18,

/mnt/vol_d/CUFE/3rd_Year/1st_term/Image_Processing/assignments/Image-proccessing/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


'\nYou should get the following test accuracies the first time \n\nSVM accuracy ~ 97.70833333333333\nKNN accuracy ~ 96.52777777777779\nNN accuracy ~ 93.95833333333333\n'

In [12]:
run_experiment('hsv_hist')
"""
You should get the following test accuracies the first time 

SVM accuracy ~ 32.083333333333336
KNN accuracy ~ 32.708333333333336
NN accuracy ~ 9.722222222222223
"""

# Why low accuracies?
# Because color histograms are not good features for digit images since digits are
# usually black and white images. Color information is not very useful here.

Loading dataset. This will take time ...
[INFO] processed 1000/7200
[INFO] processed 2000/7200
[INFO] processed 3000/7200
[INFO] processed 4000/7200
[INFO] processed 5000/7200
[INFO] processed 6000/7200
[INFO] processed 7000/7200
Finished loading dataset.
############## Training SVM ##############
SVM accuracy: 27.98611111111111 %
############## Training KNN ##############
KNN accuracy: 32.63888888888889 %
############## Training NN ##############
Iteration 1, loss = 2.20386122
Iteration 2, loss = 2.20239622
Iteration 3, loss = 2.20121294
Iteration 4, loss = 2.20024978
Iteration 5, loss = 2.19950506
Iteration 6, loss = 2.19883932
Iteration 7, loss = 2.19838523
Iteration 8, loss = 2.19798418
Iteration 9, loss = 2.19766502
Iteration 10, loss = 2.19740482
Iteration 11, loss = 2.19719360
Iteration 12, loss = 2.19704197
Iteration 13, loss = 2.19686777
Iteration 14, loss = 2.19677782
Iteration 15, loss = 2.19669375
Iteration 16, loss = 2.19659513
Iteration 17, loss = 2.19656649
Iteration 18,

/mnt/vol_d/CUFE/3rd_Year/1st_term/Image_Processing/assignments/Image-proccessing/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


'\nYou should get the following test accuracies the first time \n\nSVM accuracy ~ 32.083333333333336\nKNN accuracy ~ 32.708333333333336\nNN accuracy ~ 9.722222222222223\n'

In [13]:
run_experiment('raw')
"""
You should get the following test accuracies the first time 

SVM accuracy ~ 85.06944444444444
KNN accuracy ~ 93.95833333333333
NN accuracy ~ 88.68055555555556
"""

Loading dataset. This will take time ...
[INFO] processed 1000/7200
[INFO] processed 2000/7200
[INFO] processed 3000/7200
[INFO] processed 4000/7200
[INFO] processed 5000/7200
[INFO] processed 6000/7200
[INFO] processed 7000/7200
Finished loading dataset.
############## Training SVM ##############


/mnt/vol_d/CUFE/3rd_Year/1st_term/Image_Processing/assignments/Image-proccessing/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


SVM accuracy: 83.05555555555556 %
############## Training KNN ##############
KNN accuracy: 93.61111111111111 %
############## Training NN ##############
Iteration 1, loss = 9.09939032
Iteration 2, loss = 1.30297559
Iteration 3, loss = 1.10845431
Iteration 4, loss = 0.99257656
Iteration 5, loss = 0.81015390
Iteration 6, loss = 0.79859491
Iteration 7, loss = 0.67425904
Iteration 8, loss = 0.61163756
Iteration 9, loss = 0.61423494
Iteration 10, loss = 0.57149093
Iteration 11, loss = 0.53204749
Iteration 12, loss = 0.48477233
Iteration 13, loss = 0.45501283
Iteration 14, loss = 0.46385959
Iteration 15, loss = 0.42978143
Iteration 16, loss = 0.42926623
Iteration 17, loss = 0.44341209
Iteration 18, loss = 0.42526847
Iteration 19, loss = 0.40587795
Iteration 20, loss = 0.38949853
NN accuracy: 87.84722222222221 %


/mnt/vol_d/CUFE/3rd_Year/1st_term/Image_Processing/assignments/Image-proccessing/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


'\nYou should get the following test accuracies the first time \n\nSVM accuracy ~ 85.06944444444444\nKNN accuracy ~ 93.95833333333333\nNN accuracy ~ 88.68055555555556\n'

The classifiers list now has models trained on the last feature set you ran an experiment on. You can play around with it checking the probability it gives to each label, given an image.

In [14]:
# Example
test_img_path = r'test2.jpg'
img = cv2.imread(test_img_path)
features = extract_features(img, 'raw')  # be careful of the choice of feature set

In [15]:
nn = classifiers['NN']
nn.predict_proba([features])

array([[0.10195323, 0.17516789, 0.10081717, 0.09684416, 0.09466372,
        0.09536688, 0.11032663, 0.11413618, 0.11072415]])

In [16]:
# Grid search over a small set of hyperparameters
def grid_search_experiments(feature_set, param_grid=None):
    """Try different hyperparameters for classifiers and report accuracies.
    Returns a dict with best params and scores per model.
    """
    if param_grid is None:
        param_grid = {
            'SVM': {'C': [0.01, 0.1, 1.0]},
            'KNN': {'n_neighbors': [3,5,7,9]},
            'NN': {'hidden_layer_sizes': [(100,), (500,), (200,100)], 'alpha': [0.0001, 0.001]}
        }
    from sklearn.model_selection import GridSearchCV
    results = {}
    print(f"Running grid search on feature set: {feature_set}")
    features, labels = load_dataset(feature_set)
    X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.2, random_state=random_seed)
    for model_name in ['SVM','KNN','NN']:
        print('---', model_name)
        if model_name == 'SVM':
            base = svm.LinearSVC(random_state=random_seed, max_iter=5000)
            grid = GridSearchCV(base, param_grid={'C': param_grid['SVM']['C']}, cv=3, n_jobs=-1)
        elif model_name == 'KNN':
            base = KNeighborsClassifier()
            grid = GridSearchCV(base, param_grid={'n_neighbors': param_grid['KNN']['n_neighbors']}, cv=3, n_jobs=-1)
        else:
            base = MLPClassifier(solver='sgd', random_state=random_seed, max_iter=200, verbose=0)
            grid = GridSearchCV(base, param_grid={'hidden_layer_sizes': param_grid['NN']['hidden_layer_sizes'], 'alpha': param_grid['NN']['alpha']}, cv=3, n_jobs=-1)
        grid.fit(X_train, y_train)
        best = grid.best_estimator_
        score = best.score(X_test, y_test)
        print(f"Best {model_name}: {grid.best_params_}  test accuracy: {score*100:.2f}%")
        results[model_name] = {'best_params': grid.best_params_, 'test_accuracy': score}
    return results

In [17]:
# Run grid searches for each feature set and collect results
all_results = {}
for fs in ['hog','raw','hsv_hist']:
    try:
        res = grid_search_experiments(fs)
        all_results[fs] = res
    except Exception as e:
        print(f"Grid search failed for {fs}: {e}")
        all_results[fs] = None
print('\nSummary of grid search results:')
for fs, res in all_results.items():
    print('Feature set:', fs)
    if res:
        for m,r in res.items():
            print(f"  {m}: {r['test_accuracy']*100:.2f}%  params: {r['best_params']}")
    else:
        print('  No results')

Running grid search on feature set: hog
[INFO] processed 1000/7200
[INFO] processed 2000/7200
[INFO] processed 3000/7200
[INFO] processed 4000/7200
[INFO] processed 5000/7200
[INFO] processed 6000/7200
[INFO] processed 7000/7200
--- SVM
Best SVM: {'C': 0.1}  test accuracy: 97.57%
--- KNN
Best KNN: {'n_neighbors': 3}  test accuracy: 96.46%
--- NN


/mnt/vol_d/CUFE/3rd_Year/1st_term/Image_Processing/assignments/Image-proccessing/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/mnt/vol_d/CUFE/3rd_Year/1st_term/Image_Processing/assignments/Image-proccessing/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/mnt/vol_d/CUFE/3rd_Year/1st_term/Image_Processing/assignments/Image-proccessing/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/mnt/vol_d/CUFE/3rd_Year/1st_term/Image_Processing/assignments/Image-proccessing/.ven

Best NN: {'alpha': 0.0001, 'hidden_layer_sizes': (200, 100)}  test accuracy: 96.60%
Running grid search on feature set: raw
[INFO] processed 1000/7200
[INFO] processed 2000/7200
[INFO] processed 3000/7200
[INFO] processed 4000/7200
[INFO] processed 5000/7200
[INFO] processed 6000/7200
[INFO] processed 7000/7200
--- SVM
Best SVM: {'C': 0.01}  test accuracy: 82.71%
--- KNN
Best KNN: {'n_neighbors': 3}  test accuracy: 94.65%
--- NN
Best NN: {'alpha': 0.001, 'hidden_layer_sizes': (500,)}  test accuracy: 88.75%
Running grid search on feature set: hsv_hist
[INFO] processed 1000/7200
[INFO] processed 2000/7200
[INFO] processed 3000/7200
[INFO] processed 4000/7200
[INFO] processed 5000/7200
[INFO] processed 6000/7200
[INFO] processed 7000/7200
--- SVM
Best SVM: {'C': 1.0}  test accuracy: 27.99%
--- KNN
Best KNN: {'n_neighbors': 9}  test accuracy: 33.68%
--- NN
Best NN: {'alpha': 0.0001, 'hidden_layer_sizes': (500,)}  test accuracy: 9.86%

Summary of grid search results:
Feature set: hog
  SVM:

### Hyperparameter search — best results (from grid search)

Feature set: **hog**
- SVM: **97.57%** — params: {'C': 0.1}
- KNN: **96.46%** — params: {'n_neighbors': 3}
- NN:  **96.60%** — params: {'alpha': 0.0001, 'hidden_layer_sizes': (200, 100)}

Feature set: **raw**
- SVM: **82.71%** — params: {'C': 0.01}
- KNN: **94.65%** — params: {'n_neighbors': 3}
- NN:  **88.75%** — params: {'alpha': 0.001, 'hidden_layer_sizes': (500,)}

Feature set: **hsv_hist**
- SVM: **27.99%** — params: {'C': 1.0}
- KNN: **33.68%** — params: {'n_neighbors': 9}
- NN:  **9.86%** — params: {'alpha': 0.0001, 'hidden_layer_sizes': (500,)}

**Best overall model:** SVM on **hog** — **97.57%** (C=0.1)

Notes:
- The NN runs produced ConvergenceWarning in some configs; consider increasing `max_iter` or using the 'adam' solver for faster convergence.
- HOG features gave the best SVM/NN performance here; color histograms (`hsv_hist`) perform poorly for grayscale digit images.